In [13]:
import torch
from adaptive_ABC_KNN_KDE import AdaptiveInverseKNNKDE as SOLVER
from adaptive_ABC_KNN_KDE import UniformBoxPrior
from resources.loader_usecases import prepare_case, prepare_case_2_data
from scipy.stats import multivariate_normal
from helpers_ISRERM2026 import *

ImportError: cannot import name 'AdaptiveInverseKNNKDE' from 'adaptive_ABC_KNN_KDE' (C:\Users\roberto.rocchetta\Documents\GitHub\KNNcalibration_experimental\demo\adaptive_ABC_KNN_KDE.py)

# USECASE 1 - PARABOLOID


In [ ]:
M, Demp, Dsim = prepare_case(1, Nemp=20, Nsim=5000)
M_design, Y_emp_by_design, sim_db = adapt_case1_for_multidesign(M, Demp, Dsim)

In [ ]:
solver = SOLVER(
    model=M_design,
    Y_emp_by_design=Y_emp_by_design,
    prior=UniformBoxPrior(low=[-10, -10], high=[10, 10]),
    sim_db=sim_db,
    K=50,
    ridge=1e-3,
)

hist, diag = solver.adaptive_refine(
    max_iter=10,
    top_frac=0.3,
    n_new_per_iter=20,
    inflate=1.0,
    min_iter=3,
    target_shrink=0.01,
    improve_tol=0.05,
    patience=2,
)

print(" 🎬 Generating posterior progression plot...")

print("\n🎯 SUMMARY:")
print(f"• Initial mean radius: {hist['mean_radius'].iloc[0]:.3f} → Final: {hist['mean_radius'].iloc[-1]:.3f}")
if "max_radius" in hist.columns:
    print(f"• Initial max radius: {hist['max_radius'].iloc[0]:.3f} → Final: {hist['max_radius'].iloc[-1]:.3f}")

if "db_size" in hist.columns:
    print(f"• Database: {hist['db_size'].iloc[0]} → {hist['db_size'].iloc[-1]} samples")
print(f"• Iterations: {len(hist)} (stopped: {hist['stop_reason'].iloc[-1]})")

mode_x, mode_p = solver.posterior_mode_from_db()
print(f"• Final posterior mode: {mode_x}")

In [ ]:
plot_posterior_vs_true_theta_case1(solver, Demp)
plot_posterior_predictive_vs_empirical(solver, n_post_samples=3000)
plot_empirical_vs_posterior_intervals(solver, n_post_samples=3000)

In [ ]:
solver.fit_local_models()
plot_posterior_x_by_design_case1(solver)


In [ ]:
plot_posterior_vs_true_theta_by_design_case1(solver, Demp)

# USECASE 2 - AIRMODE

In [ ]:

# ============================================================
# AIRMODE calibration workflow
# ============================================================
model_airmode, Y_emp_by_design, sim_db, prior = prepare_airmode_for_calibration(
    Nemp=30,
    Nsim=1000,
)

solver = SOLVER(
    model=model_airmode,
    Y_emp_by_design=Y_emp_by_design,
    prior=prior,
    sim_db=sim_db,
    K=20,
    ridge=1e-2,
    seed=123,
)

hist, diag = solver.adaptive_refine(
    max_iter=50,
    top_frac=0.1,
    n_new_per_iter=20,
    inflate=1.0,
    min_iter=20,
    target_shrink=0.0005,
    improve_tol=0.001,
    patience=3,
)

print(hist)
print(diag.head())


In [ ]:
print("AIRMODE SUMMARY")
print(f"Initial mean radius: {hist['mean_radius'].iloc[0]:.4f}")
print(f"Final mean radius:   {hist['mean_radius'].iloc[-1]:.4f}")
print(f"Iterations:          {len(hist)}")
print(f"Stop reason:         {hist['stop_reason'].iloc[-1]}")

db_size_initial = sim_db["X"].shape[0]
db_size_final = solver.X_db.shape[0]
print(f"Database size:       {db_size_initial} -> {db_size_final}")


In [ ]:
from resources.AIRMODE.load_helpers import *
# ---------- Paths ----------
ref_tmcmc_data_path = "../resources/AIRMODE/data/reference_TMCMC_DLRAirmod.mat"

# ---------- Load & extract TMCMC
tmobj = load_mat_any(str(ref_tmcmc_data_path))
theta_tmcmc, src_path = find_theta_matrix_from_mat(tmobj)
print(f"[TMCMC] extracted from '{src_path}' -> {theta_tmcmc.shape}")


X_post, Y_post = plot_airmode_posterior_style(
    solver=solver,
    theta_tmcmc=theta_tmcmc,
    theta_latex_names=[ r"$\theta_1$",  r"$\theta_2$", r"$\theta_3$",    r"$\theta_4$",  r"$\theta_5$", r"$\theta_6$",
                      r"$\theta_7$",  r"$\theta_8$", r"$\theta_9$",   r"$\theta_{10}$",  r"$\theta_{11}$"],
    Y_latex_names=[ r"$D_1$",  r"$D_2$", r"$D_3$",  r"$D_4$",  r"$D_5$", r"$D_6$",   r"$D_7$",  r"$D_8$", r"$D_9$",   r"$D_{10}$"],
    pairs_2_plt=[[6, 5], [3, 2], [5, 3], [1, 9]],
    n2plt=10_000,
    n_post=10_000,
    design="AIRMODE",
)